# SCOPUS Collection Pipeline

*Author: Regina Chua*

> This notebook is the SCOPUS arm of the systematic review. It mirrors `pubmed.ipynb` so the two
> pipelines stay comparable: the same `(disease) AND (spatial) AND (exposure) NOT (exclusions)`
> logic is reused, only translated into SCOPUS' Advanced Search syntax. Everything that defines
> *what* we search for is imported from `search_strategy.py` so the query stays in sync with every
> other database.

SCOPUS exposes an official API (unlike Google Scholar), accessed here through the
[`pybliometrics`](https://pybliometrics.readthedocs.io) wrapper. An Elsevier API key is required;
`pybliometrics.init()` reads it from the saved configuration created by `pybliometrics.init()` the
first time it is run.

**Access note:** abstract text (`description`) and author keywords (`authkeywords`) are only
returned under the `COMPLETE` view, which requires a subscriber token (typically on an
institutional network). The `STANDARD` view used by default still returns titles, DOIs, journals,
and citation counts, so the pipeline runs either way.

**References:** SCOPUS Search API via [`pybliometrics`](https://github.com/pybliometrics-dev/pybliometrics).

## 1. Environment Setup

> Same core libraries as the PubMed notebook plus `pybliometrics`. `pybliometrics.init()` loads the
> stored Elsevier API key from the pybliometrics config file (`~/.config/pybliometrics.cfg`). After
> initialising I read the config back and print the masked key so it's easy to confirm the right key
> is being used, and whether an InstToken is present. An InstToken is needed for subscriber-level
> access (full pagination, `COMPLETE` view); without one the API caps at 5000 results per query —
> which the year-splitting in Section 3 handles automatically.

In [1]:
from datetime import datetime
from pathlib import Path
import re

import pandas as pd

import pybliometrics
from pybliometrics.scopus import ScopusSearch
from pybliometrics.utils.startup import get_config

from search_strategy import (
    INCLUSION_CRITERIA,
    ALTERNATE_TERMS,
    EXCLUSION_TERMS,
    DATE_FILTER,
    CLEANING_RULES,
)

pd.set_option("display.max_colwidth", 120)

SCOPUS_READY = False
try:
    pybliometrics.init()  # reads the saved Elsevier API key from ~/.config/pybliometrics.cfg
    SCOPUS_READY = True
    print("pybliometrics initialised OK.")
except Exception as exc:  # noqa: BLE001
    print(
        "pybliometrics could not initialise:\n"
        f"  {type(exc).__name__}: {exc}\n\n"
        "Run `python -c \"import pybliometrics; pybliometrics.init()\"` once and paste in your\n"
        "Elsevier API key (https://dev.elsevier.com/apikey/manage), then re-run this cell."
    )

# --- Confirm which key and token are loaded ---
if SCOPUS_READY:
    try:
        cfg = get_config()
        api_key   = cfg.get("Authentication", "APIKey",    fallback=None)
        inst_token = cfg.get("Authentication", "InstToken", fallback=None)
        if api_key:
            masked = api_key[:4] + "..." + api_key[-4:]
            print(f"  API key confirmed: {masked}")
        else:
            print("  WARNING: No API key found in config — re-run pybliometrics.init().")
        print(f"  InstToken set: {bool(inst_token)}")
        if not inst_token:
            print(
                "\n  Without an InstToken the API is limited to 5,000 results per query.\n"
                "  The collection cell splits by year to stay within this limit.\n"
                "  For full subscriber access, ask your library for the InstToken and add:\n"
                "    InstToken = <token>\n"
                "  under [Authentication] in ~/.config/pybliometrics.cfg"
            )
    except Exception as exc:  # noqa: BLE001
        print(f"  Could not read config details: {exc}")

print("\nEnvironment ready.")

## 2. Build Query

> Here I translate the shared criteria into SCOPUS syntax. The PubMed notebook uses `[TIAB]` field
> tags; SCOPUS' equivalent is `TITLE-ABS-KEY(...)`, which searches title, abstract, and keywords in
> one go. The boolean shape is identical to PubMed — three `AND`-joined inclusion groups followed by
> `AND NOT` for the exclusions — and the 2020–2025 window is expressed with `PUBYEAR` bounds.
>
> As in PubMed I fold the NLP-surfaced `ALTERNATE_TERMS` into the inclusion groups; set
> `INCLUDE_ALTERNATE_TERMS = False` to compare against the core criteria only. SCOPUS supports the
> `*` wildcard, so terms like `parkinson* disease` carry over unchanged.

In [2]:
def merge_terms(primary, alternates):
    """Combine primary inclusion terms with alternate/NLP terms per category
    (order-preserving, case-insensitive de-duplication). Mirrors pubmed.ipynb."""
    merged = {}
    for category, base_terms in primary.items():
        seen, combined = set(), []
        for t in list(base_terms) + list(alternates.get(category, [])):
            if t.lower() not in seen:
                seen.add(t.lower())
                combined.append(t)
        merged[category] = combined
    return merged


def tak_group(terms):
    """Wrap terms in a SCOPUS TITLE-ABS-KEY() OR-group."""
    return "TITLE-ABS-KEY(" + " OR ".join(f'\"{t}\"' for t in terms) + ")"


def build_scopus_query(inclusion, exclusion, date_filter):
    """Compose the SCOPUS Advanced Search query.

    Structure: TITLE-ABS-KEY(disease) AND TITLE-ABS-KEY(spatial)
               AND TITLE-ABS-KEY(exposure) AND NOT TITLE-ABS-KEY(exclusions)
               AND PUBYEAR > start-1 [AND PUBYEAR < end+1]
    """
    include = " AND ".join([
        tak_group(inclusion["disease"]),
        tak_group(inclusion["spatial"]),
        tak_group(inclusion["exposure"]),
    ])
    q = f"{include} AND NOT {tak_group(exclusion)}"

    # PUBYEAR is exclusive on both sides, so widen by one year each way.
    start_year = int(date_filter["start_date"][:4])
    q += f" AND PUBYEAR > {start_year - 1}"
    if date_filter.get("end_date"):
        end_year = int(date_filter["end_date"][:4])
        q += f" AND PUBYEAR < {end_year + 1}"
    return q


INCLUDE_ALTERNATE_TERMS = True
search_criteria = (
    merge_terms(INCLUSION_CRITERIA, ALTERNATE_TERMS)
    if INCLUDE_ALTERNATE_TERMS
    else INCLUSION_CRITERIA
)

query = build_scopus_query(search_criteria, EXCLUSION_TERMS, DATE_FILTER)

print("Alternate terms folded into query:", INCLUDE_ALTERNATE_TERMS)
for category, term_list in search_criteria.items():
    print(f"{category} terms ({len(term_list)}):", term_list)
print("\nQuery:\n", query)

## 3. Collect Articles from SCOPUS

> **Root cause of the `Scopus400Error`:** `pybliometrics` defaults to `subscriber=True`, which uses
> cursor-based navigation and requests the maximum records per page that a subscriber account is
> allowed. Without an InstToken the API rejects that page-size with a 400.
>
> **Fix — two changes:**
> 1. `subscriber=False` — switches to page-based navigation with 25 records/page, which the free
>    API key supports. The trade-off is a hard cap of 5,000 results per query.
> 2. **Year-by-year splitting** — running one query per year (2020, 2021, …, 2025) keeps each
>    slice well under 5,000, so we get every record across the full window. Results are combined
>    and deduplicated after collection.
>
> Once you have an InstToken from your institution, set `SUBSCRIBER_ACCESS = True` and the
> collector switches back to uncapped cursor pagination and can also use `VIEW = "COMPLETE"` to
> retrieve abstracts and author keywords.

In [3]:
# --- Configuration ---
# Set to True once you have an InstToken — enables uncapped pagination and COMPLETE view.
SUBSCRIBER_ACCESS = False
VIEW = "COMPLETE" if SUBSCRIBER_ACCESS else "STANDARD"


def flatten_scopus(rec):
    """Flatten a ScopusSearch result namedtuple into our shared schema."""
    doi = getattr(rec, "doi", None)
    return {
        "title":            getattr(rec, "title", None),
        "abstract":         getattr(rec, "description", None),   # COMPLETE view only
        "publication_date": getattr(rec, "coverDate", None),
        "authors":          getattr(rec, "author_names", None),  # ';'-separated
        "journal":          getattr(rec, "publicationName", None),
        "doi":              doi,
        "url":              f"https://doi.org/{doi}" if doi else None,
        "num_citations":    getattr(rec, "citedby_count", None),
        "pubmed_id":        getattr(rec, "pubmed_id", None),
        "keywords":         getattr(rec, "authkeywords", None),  # COMPLETE view only
        "source":           "scopus",
    }


def year_query(base_q, year):
    """Replace the PUBYEAR range in the base query with a single-year constraint."""
    q = re.sub(r"\s+AND\s+PUBYEAR\s*[><]\s*\d+", "", base_q)
    return f"{q} AND PUBYEAR > {year - 1} AND PUBYEAR < {year + 1}"


run_ts = datetime.now().isoformat(timespec="seconds")
all_records = []

if not SCOPUS_READY:
    print("Skipping — pybliometrics not initialised (see Section 1).")
else:
    start_year = int(DATE_FILTER["start_date"][:4])
    end_year   = int(DATE_FILTER["end_date"][:4]) if DATE_FILTER.get("end_date") else datetime.now().year
    years = list(range(start_year, end_year + 1))

    print(f"Collecting year by year ({start_year}–{end_year}), subscriber={SUBSCRIBER_ACCESS}, view={VIEW}")
    print(f"Starting at {run_ts}\n")

    for yr in years:
        yr_q = year_query(query, yr)
        try:
            probe = ScopusSearch(yr_q, view=VIEW, download=False, subscriber=SUBSCRIBER_ACCESS)
            n = probe.get_results_size()
            print(f"  {yr}: {n} results", end="")
            if n == 0:
                print(" — skipped")
                continue
            if n > 5000 and not SUBSCRIBER_ACCESS:
                print(f" — WARNING: {n} exceeds the 5,000 non-subscriber cap; only first 5,000 retrieved")
            search = ScopusSearch(yr_q, view=VIEW, download=True, verbose=False,
                                  subscriber=SUBSCRIBER_ACCESS)
            results = search.results or []
            all_records.extend(flatten_scopus(r) for r in results)
            print(f" — downloaded {len(results)}")
        except Exception as exc:  # noqa: BLE001
            print(f" — FAILED ({type(exc).__name__}: {exc})")

    df_raw = pd.DataFrame(all_records)
    print(f"\nTotal collected: {len(df_raw)} records across {len(years)} years.")

preview_cols = [c for c in ["title", "publication_date", "journal", "doi", "num_citations"]
                if c in df_raw.columns]
if not df_raw.empty:
    display(df_raw[preview_cols].head())

## 4. Clean Results

> Same cleaning contract as PubMed, driven by `CLEANING_RULES`: drop duplicate titles and (by
> default) require a DOI so every record is uniquely identifiable for cross-database deduplication
> later. SCOPUS does not always populate `pubmed_id`, so unlike PubMed I do **not** require it here.

In [ ]:
df_clean = df_raw.copy()

if not df_clean.empty:
    if CLEANING_RULES.get("remove_duplicate_titles", True):
        df_clean = df_clean.dropna(subset=["title"])
        df_clean["_title_lower"] = df_clean["title"].str.lower().str.strip()
        df_clean = df_clean.drop_duplicates(subset=["_title_lower"]).drop(columns=["_title_lower"])
    if CLEANING_RULES.get("require_doi", True) and "doi" in df_clean.columns:
        df_clean = df_clean.dropna(subset=["doi"])

print(f"Records after cleaning: {len(df_clean)}  (from {len(df_raw)} raw)")
if not df_clean.empty:
    display(df_clean[preview_cols].head())

## 5. Export

> Save the cleaned set to CSV — the stable hand-off artifact for the screening stage and for merging
> with PubMed/EMBASE/Web of Science in the deduplication step (Milestone 4). I only write the file
> when there is something to write, so a failed/blocked run doesn't clobber a previous good export.

In [ ]:
output_path = Path("scopus_results_2026.csv")

if df_clean.empty:
    print("Nothing to export — df_clean is empty (see Sections 1 and 3).")
else:
    df_clean.to_csv(output_path, index=False)
    print(f"Exported {len(df_clean)} records to {output_path.resolve()}")
    print(f"Run timestamp: {run_ts}")

## 6. Notes & Next Steps

> - **Subscriber access / InstToken:** to lift the 5,000-per-year cap and get abstracts + keywords
>   (`COMPLETE` view), ask your institution's library for an InstToken. Add it to the pybliometrics
>   config at `~/.config/pybliometrics.cfg` under `[Authentication]` as `InstToken = <token>`,
>   then set `SUBSCRIBER_ACCESS = True` in Section 3.
> - **Quota:** the Scopus API has a weekly quota. `pybliometrics` caches each year-slice to disk,
>   so re-running is free for previously downloaded slices. Force a fresh pull with `refresh=True`.
> - **Abstracts for screening:** the `STANDARD` view does not return abstract text. Before running
>   Milestone 2 LLM pre-screening on SCOPUS records, re-run with `SUBSCRIBER_ACCESS = True` and
>   `VIEW = "COMPLETE"` (set automatically once `SUBSCRIBER_ACCESS` is flipped).
> - **Deduplication:** export columns match the other database CSVs for clean concatenation in
>   Milestone 4.
> - **Query parity:** if `search_strategy.py` changes, re-run all collection notebooks so every
>   database reflects the same criteria.